In [1]:
#PARAMETERS  ← tag this cell
#batch_id = "MANUAL_RUN"
excel_path = "Files/landing/EXCEL"

StatementMeta(, b8ad9fd1-d830-43f0-90a9-2d6541f915c5, 3, Finished, Available, Finished, False)

In [2]:
from datetime import datetime, timezone
import openpyxl
from pyspark.sql import functions as F

run_ts = datetime.now(timezone.utc)
load_date = run_ts.date()

# These workbooks carry a title block in rows 1-3 and headers in row 4,
# because that is what human-maintained spreadsheets look like. Reading from
# row 1 gives you a header of 'Medical Staff Roster' and one usable column.
HEADER_ROW = 4

FILES = {
    "doctor_roster.xlsx": ("HR", "doctor", "hr_doctor"),
    "survey_response_export.xlsx": ("SURVEY", "survey_response", "survey_response"),
}

StatementMeta(, b8ad9fd1-d830-43f0-90a9-2d6541f915c5, 4, Finished, Available, Finished, False)

In [3]:

from datetime import datetime, timezone
import openpyxl
from pyspark.sql import functions as F

run_ts = datetime.now(timezone.utc)
load_date = run_ts.date()

# These workbooks carry a title block in rows 1-3 and headers in row 4,
# because that is what human-maintained spreadsheets look like. Reading from
# row 1 gives you a header of 'Medical Staff Roster' and one usable column.
HEADER_ROW = 4

FILES = {
    "doctor_roster.xlsx": ("HR", "doctor", "hr_doctor"),
    "survey_response_export.xlsx": ("SURVEY", "survey_response", "survey_response"),
}

StatementMeta(, b8ad9fd1-d830-43f0-90a9-2d6541f915c5, 5, Finished, Available, Finished, False)

In [5]:

for fname, (source_system, entity_name, target_table) in FILES.items():
    local = f"/lakehouse/default/Files/landing/EXCEL/{fname}"
    wb = openpyxl.load_workbook(local, read_only=True, data_only=True)
    ws = wb.active

    rows = list(ws.iter_rows(min_row=HEADER_ROW, values_only=True))
    header = [str(c).strip() if c is not None else f"col_{i}"
              for i, c in enumerate(rows[0])]
    body = [r for r in rows[1:] if any(v is not None for v in r)]

    # Everything as string. Excel stores numeric-looking text as numbers, so
    # a doctor id of 00123 arrives as 123 and every downstream join fails
    # silently. Casting happens in Silver where a bad value can be
    # quarantined with a rule id attached.
    data = [tuple("" if v is None else str(v) for v in r) for r in body]

    df = spark.createDataFrame(data, schema=header)
    df = (df.withColumn("_source_system", F.lit(source_system))
            .withColumn("_entity_name", F.lit(entity_name))
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn("_ingest_ts", F.lit(run_ts))
            .withColumn("_load_date", F.lit(load_date)))

    payload = [c for c in df.columns if not c.startswith("_")]
    df = df.withColumn("_row_hash", F.sha2(F.concat_ws("||", *[
        F.coalesce(F.upper(F.trim(F.col(c).cast("string"))), F.lit("<NULL>"))
        for c in payload]), 256))

    if spark.catalog.tableExists(target_table):
        spark.sql(f"DELETE FROM {target_table} WHERE _load_date = '{load_date}'")

    (df.write.format("delta").mode("append")
       .partitionBy("_load_date").option("mergeSchema", "true")
       .saveAsTable(target_table))

    print(f"{target_table:<24} {df.count():>7,} rows, {len(header)} columns")
    wb.close()

StatementMeta(, b8ad9fd1-d830-43f0-90a9-2d6541f915c5, 7, Finished, Available, Finished, False)

hr_doctor                    240 rows, 14 columns
survey_response            8,499 rows, 18 columns
